In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

# Load training data to fit preprocessors
print("Loading training data...")
train_transaction = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')

train_data = train_transaction.merge(train_identity, how='left', on='TransactionID')

y_train = train_data['isFraud']
X_train = train_data.drop(['isFraud', 'TransactionID'], axis=1)

# Identify column types
cat_cols = X_train.select_dtypes(include='object').columns
num_cols = X_train.select_dtypes(exclude='object').columns

# Handle missing values in training data
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_train[num_cols] = X_train[num_cols].fillna(X_train[num_cols].median())

# Fit encoders and scalers on training data
print("Fitting preprocessors...")
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
enc.fit(X_train[cat_cols])

scaler = StandardScaler()
scaler.fit(X_train[num_cols])

# Store median values for test imputation
num_medians = X_train[num_cols].median()

# Load test data
print("Loading test data...")
test_transaction = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_identity.csv')

test_data = test_transaction.merge(test_identity, how='left', on='TransactionID')
test_ids = test_data['TransactionID']
X_test = test_data.drop(['TransactionID'], axis=1)

# Align test columns with training columns
print("Aligning columns...")
# Add missing columns with NaN
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = np.nan

# Remove extra columns not in training
X_test = X_test[X_train.columns]

# Update cat_cols and num_cols based on actual test data
cat_cols_test = [col for col in cat_cols if col in X_test.columns]
num_cols_test = [col for col in num_cols if col in X_test.columns]

# Preprocess test data using training statistics
print("Preprocessing test data...")
X_test[cat_cols_test] = X_test[cat_cols_test].fillna('missing')
X_test[num_cols_test] = X_test[num_cols_test].fillna(num_medians[num_cols_test])

X_test[cat_cols_test] = enc.transform(X_test[cat_cols_test])
X_test[num_cols_test] = scaler.transform(X_test[num_cols_test])
X_test[num_cols_test] = X_test[num_cols_test].astype('float32')

# Build the same model architecture
print("Building model...")
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_test.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['AUC']
)

# You need to train the model first or load weights
# If you've saved the model, load it like this:
# model = tf.keras.models.load_model('fraud_detection_model.h5')

# For now, let's train it (you should ideally save and load the trained model)
print("Training model on full training data...")
from sklearn.utils.class_weight import compute_class_weight

# Prepare training data
X_train[cat_cols] = enc.transform(X_train[cat_cols])
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_train[num_cols] = X_train[num_cols].astype('float32')

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=2048,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

# Make predictions on test data
print("Making predictions...")
test_preds = model.predict(X_test, batch_size=2048, verbose=1).ravel()

# Create submission file
submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission file created successfully!")
print(f"Predictions range: [{test_preds.min():.4f}, {test_preds.max():.4f}]")
print(f"Mean prediction: {test_preds.mean():.4f}")
print(submission.head())

Loading training data...
Fitting preprocessors...
Loading test data...
Aligning columns...
Preprocessing test data...
Building model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1768584683.432644      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1768584683.436439      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Training model on full training data...
Epoch 1/15


I0000 00:00:1768584699.662863     124 service.cc:152] XLA service 0x7c49d8029370 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768584699.662900     124 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1768584699.662904     124 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1768584700.320925     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


 27/289 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.6271 - loss: 0.8013

I0000 00:00:1768584704.197693     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


289/289 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - AUC: 0.7408 - loss: 0.6462
Epoch 2/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.8253 - loss: 0.5148
Epoch 3/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8401 - loss: 0.4940
Epoch 4/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8480 - loss: 0.4812
Epoch 5/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8542 - loss: 0.4726
Epoch 6/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8565 - loss: 0.4674
Epoch 7/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8622 - loss: 0.4565
Epoch 8/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.8665 - loss: 0.4511
Epoch 9/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8689 - loss: 0.4474
Epoch 10/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8680 - loss: 0.4481
Epoch 11/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8697 - loss: 0.4455
Epoch 12/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.8706 - loss: 0.4431
Epoch 13/15
289/289 ━━━━━━━━━━━━━━